# Lesson 12 - Reduksjon av samtalehistorikk med Agent Scratchpad

Denne notisboken demonstrerer hvordan man håndterer kontekst i lange samtaler ved bruk av Microsoft Agent Framework. Etter hvert som samtalene vokser, øker antallet tokens – til slutt overstiger de modellens kontekstvindu. Vi takler dette med et **mønster for kontekstoppsummering** og en **agent scratchpad** for vedvarende minne.

## Hva du vil lære:
1. **Hvorfor kontekststyring er viktig**: Forstå tokenbegrensninger og kontekstvinduer
2. **Kontekstbevisste agenter**: Bygge agenter som styrer sin egen samtalekontekst
3. **Mønster for kontekstoppsummering**: Bruke verktøy for å kondensere samtalehistorikk
4. **Agent Scratchpad**: Vedvarende minne som overlever kontekstreduksjon

## Forutsetninger:
- Azure OpenAI-oppsett med miljøvariabler konfigurert
- Forståelse av grunnleggende agentkonsepter fra tidligere leksjoner


## Oppsett


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Hvorfor kontekststyring er viktig

Hver LLM har et begrenset **kontekstvindu** — det maksimale antallet tokens den kan behandle i en enkelt forespørsel. Etter hvert som en samtale med flere runder utvikler seg:

- **Antall tokens øker lineært** med hver brukerbeskjed og assistentsvar.
- **Prompt-tokens dominerer kostnadene** fordi hele historikken sendes på nytt hver runde.
- Til slutt **overskrider samtalen kontekstvinduet** og modellen enten forkorter eller gir feil.

### Strategier for å håndtere kontekst

| Strategi | Hvordan det fungerer | Fordeling |
|---|---|---|
| **Forkorting** | Dropper eldste meldinger | Mister tidlig kontekst |
| **Oppsummering** | Komprimerer eldre meldinger til en oppsummering | Noe detalj går tapt, men viktige punkter beholdes |
| **Notatblokk / Ekstern minne** | Lagrer viktige fakta utenfor samtalen | Krever verktøykall, men overlever enhver reduksjon |

I denne notatboken kombinerer vi **oppsummering** med et **notatblokkverktøy** slik at agenten kan opprettholde kontinuitet selv når samtalehistorikken er kondensert.


## Lage en kontekstbevisst agent


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulering av en lang samtale

La oss gå gjennom en samtale med flere runder for å se hvordan kontekst akkumuleres. Agenten skal beholde viktige detaljer (preferanser, budsjett, reisedatoer) gjennom samtalene og vise kontinuitet.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Merk hvordan agenten beholder konteksten fra tidligere runder — den vet om Japan, sushi, templer, fotografering, budsjettet på 3000 dollar, soloreise og tidsrammen i april. I en kort samtale fungerer dette bra, men etter hvert som samtalen vokser, blir det kostbart å sende hele historikken på nytt.

La oss fortsette samtalen med flere runder for å se hvordan konteksten akkumuleres:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Oppsummeringsmønster

Etter hvert som samtalen vokser, kan vi bruke et **oppsummeringsverktøy** for å kondensere akkumulert kontekst til et kompakt format. Agenten kaller dette verktøyet for å registrere viktige preferanser slik at selv om eldre meldinger fjernes, beholdes den essensielle informasjonen.

Dette mønsteret er byggeklossen for mer sofistikert historikkreduksjon:
1. Agenten identifiserer viktige fakta fra samtalen
2. Den kaller oppsummeringsverktøyet for å lagre dem
3. Eldre meldinger kan trygt fjernes fordi oppsummeringen fanger det som er viktig

Nedenfor definerer vi et `summarize_preferences` verktøy som agenten kan kalle for å registrere en kompakt oppsummering av det den har lært.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Sammendrag

I denne leksjonen lærte du hvordan du håndterer kontekst i langvarige samtaler med agenter ved hjelp av Microsoft Agent Framework:

### Nøkkelkonsepter
- **Kontekstvinduer er begrensede** — hver token i samtalehistorikken koster penger og teller mot grensen.
- **Sammendragsverktøy** lar agenten kondensere akkumulert kontekst til kompakte sammendrag, som reduserer token-bruk samtidig som essensiell informasjon bevares.
- **Agent scratchpads** gir vedvarende ekstern hukommelse som overlever alle samtalereduksjoner.

### Hva du bygde
- En **kontekstbevisst agent** som opprettholder kontinuitet over flereturnsamtaler
- Et **sammendragsverktøy** (`summarize_preferences`) som registrerer viktige brukeropplysninger i et kompakt format
- En **flereturnsamtale** som demonstrerer kontekstbevaring og håndtering av endringer

### Virkelige bruksområder
- **Kundeserviceroboter**: Husker preferanser over lange supportsamtaler
- **Personlige assistenter**: Følger pågående prosjekter uten å måtte forklare kontekst på nytt
- **Utdanningstutorer**: Opprettholder studentens fremgang over mange interaksjoner

### Neste steg
- Implementere et fullverdig scratchpad-verktøy med filbasert vedvarende lagring
- Legge til automatisk historikkforkorting etter sammendrag
- Kombinere med vektordatabaser for semantisk hukommelsessøk
- Bygge agenter som kan gjenoppta samtaler dager senere med full kontekst


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved bruk av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vennligst vær oppmerksom på at automatiserte oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket bør anses som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som følge av bruken av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
